# pyGC Benchmarks: Spectral GC Pipeline vs. Data Size

Measures **median wall-clock time** of the full spectral Granger Causality pipeline:

$$\text{data} \xrightarrow{\text{CSD}} S(f) \xrightarrow{\text{Wilson}} H, Z \xrightarrow{\text{GC}} I_{x \to y}(f)$$

Benchmark dimensions:
| Axis | Options |
|---|---|
| **Spectral method** (→ S) | FFT, Welch, Morlet wavelet, Parametric (Yule-Walker) |
| **Wilson backend** | NumPy, JAX (JIT-compiled, if installed) |
| **Data size** | N ∈ {256, 512, 1024, 2048, 4096} time samples |

In [1]:
import math
import time
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal as sp_signal

warnings.filterwarnings('ignore')

from pygc import (
    YuleWalker, compute_transfer_function,
    wilson_factorization, wilson_factorization_jax,
    granger_causality, JAX_AVAILABLE, JAX_FLOAT64,
)
from pygc.spectral_analysis.fourier import compute_freq, csd_fourier, morlet_transform
from pygc.spectral_analysis.time_frequency import welch_spectrum

plt.rcParams.update({'figure.dpi': 100, 'font.size': 12})

# ── Configuration ────────────────────────────────────────────────────────────
Fs        = 200                            # sampling rate (Hz)
AR_ORDER  = 10                             # Yule-Walker model order
N_REPEATS = 3                              # timed repetitions (median)
N_VALUES  = np.arange(2**10, 2**14, 256)#[256, 512, 1024, 2048, 4096] # data sizes to sweep
N_CYCLES  = 7.0                            # Morlet wavelet cycles

# MNE Morlet kernel length ≈ 10 * n_cycles / (2π * f) * Fs.
# The kernel must be shorter than the smallest N in N_VALUES, so:
#   f_min > 10 * n_cycles * Fs / (2π * N_min)
_f_min = math.ceil(10 * N_CYCLES * Fs / (2 * math.pi * min(N_VALUES))) + 1
MORLET_FREQS = np.linspace(_f_min, Fs / 2 - 1, 80)  # fixed 80-freq Morlet grid
print(f"Morlet freq range: {MORLET_FREQS[0]:.1f} – {MORLET_FREQS[-1]:.1f} Hz  ({len(MORLET_FREQS)} bins)")

Morlet freq range: 4.0 – 99.0 Hz  (80 bins)


## JAX availability

In [2]:
from pygc import JAX_AVAILABLE, JAX_FLOAT64

if JAX_AVAILABLE:
    import jax
    devices   = jax.devices()
    platform  = jax.default_backend()
    precision = "complex128 (float64)" if JAX_FLOAT64 else "complex64 (float32)"
    print(f"JAX {jax.__version__}")
    print(f"  platform : {platform}")
    print(f"  devices  : {devices}")
    print(f"  precision: {precision}")
    if not JAX_FLOAT64:
        print("\n  ⚠  Metal GPU detected — running in float32 (lower precision, faster).")
    print("\nJAX benchmarks will be included.")
else:
    print("JAX not installed — JAX benchmarks skipped.")
    print()
    print("Install options:")
    print("  CPU / NVIDIA:       pip install jax")
    print("  Apple Silicon GPU:  pip install jax-metal")

JAX 0.4.35
  platform : cpu
  devices  : [CpuDevice(id=0)]
  precision: complex128 (float64)

JAX benchmarks will be included.


## Helper functions

In [3]:
# ── Data ─────────────────────────────────────────────────────────────────────

def make_data(N, seed=0):
    """2-channel white Gaussian noise, shape (2, N)."""
    return np.random.default_rng(seed).standard_normal((2, N))


# ── Timing utility ────────────────────────────────────────────────────────────

def time_fn(fn, n=N_REPEATS):
    """Median wall-clock time (s) over n calls."""
    t = []
    for _ in range(n):
        t0 = time.perf_counter()
        fn()
        t.append(time.perf_counter() - t0)
    return float(np.median(t))


# ── CSD builders (return S of shape (2, 2, n_freq) and freq axis) ─────────────
# Still used in cell 12 to time Wilson factorization independently.

def build_S_fft(data):
    """FFT periodogram CSD.
    n_freq = N//2 + 1 — scales with N.
    """
    x, y = data[0], data[1]
    f = compute_freq(len(x), Fs)
    S = np.zeros((2, 2, len(f)), dtype=complex)
    S[0, 0] = csd_fourier(x, x, f, Fs)
    S[0, 1] = csd_fourier(x, y, f, Fs)
    S[1, 0] = np.conj(S[0, 1])
    S[1, 1] = csd_fourier(y, y, f, Fs)
    return S, f


def build_S_welch(data):
    """Welch-averaged CSD via scipy.signal (default nperseg=256).
    n_freq = nperseg//2 + 1 = 129 — independent of N for N >= 256.
    """
    S = welch_spectrum(data[np.newaxis], fs=Fs)      # (2, 2, n_freq)
    f, _ = sp_signal.csd(data[0], data[0], Fs)      # probe matching freq axis
    return S, f


def build_S_morlet(data):
    """Morlet-wavelet CSD, time-averaged.
    n_freq = len(MORLET_FREQS) = 80 — independent of N.
    """
    x, y = data[0], data[1]
    N = len(x)
    Wx = morlet_transform(x, MORLET_FREQS, Fs, n_cycles=N_CYCLES)   # (N, 80)
    Wy = morlet_transform(y, MORLET_FREQS, Fs, n_cycles=N_CYCLES)
    S = np.zeros((2, 2, len(MORLET_FREQS)), dtype=complex)
    S[0, 0] = (Wx * np.conj(Wx) / N).mean(axis=0)
    S[0, 1] = (Wx * np.conj(Wy) / N).mean(axis=0)
    S[1, 0] = np.conj(S[0, 1])
    S[1, 1] = (Wy * np.conj(Wy) / N).mean(axis=0)
    return S, MORLET_FREQS


BUILDERS = [
    ('FFT',    build_S_fft),
    ('Welch',  build_S_welch),
    ('Morlet', build_S_morlet),
]

# spectral_method name used by granger_causality
SPECTRAL_METHOD_MAP = {'FFT': 'fourier', 'Welch': 'welch', 'Morlet': 'morlet'}


# ── GC pipeline runners (integrated: CSD + Wilson + GC in one call) ───────────

def _spectral_params(spectral_method):
    if spectral_method == 'morlet':
        return {'freqs': MORLET_FREQS, 'n_cycles': N_CYCLES}
    return None


def run_numpy(data, spectral_method):
    """Wilson (NumPy) + GC."""
    return granger_causality(data, Fs, spectral_method=spectral_method,
                              backend='numpy', verbose=False,
                              spectral_params=_spectral_params(spectral_method))


def run_jax(data, spectral_method):
    """Wilson (JAX JIT) + GC."""
    return granger_causality(data, Fs, spectral_method=spectral_method,
                              backend='jax',
                              spectral_params=_spectral_params(spectral_method))


def run_parametric(data):
    """Yule-Walker CSD + Wilson (NumPy) + GC."""
    f = compute_freq(data.shape[1], Fs)
    AR, sigma = YuleWalker(data, AR_ORDER)
    _, S = compute_transfer_function(AR, sigma, f, Fs)
    _, H, Z = wilson_factorization(S, f, Fs, verbose=False)
    Hxx, Hxy, Hyx, Hyy = H[0, 0], H[0, 1], H[1, 0], H[1, 1]
    Hxx_t = Hxx + (Z[0, 1] / Z[0, 0]) * Hxy
    Hyy_c = Hyy + (Z[1, 0] / Z[1, 1]) * Hyx
    Ix2y = np.log(
        (Hyy_c * Z[1, 1] * np.conj(Hyy_c)
         + Hyx * (Z[0, 0] - Z[1, 0]**2 / Z[1, 1]) * np.conj(Hyx))
        / (Hyy_c * Z[1, 1] * np.conj(Hyy_c))
    ).real
    Iy2x = np.log(
        (Hxx_t * Z[0, 0] * np.conj(Hxx_t)
         + Hxy * (Z[1, 1] - Z[0, 1]**2 / Z[0, 0]) * np.conj(Hxy))
        / (Hxx_t * Z[0, 0] * np.conj(Hxx_t))
    ).real
    det_S = np.linalg.det(S.transpose(2, 0, 1))
    Ixy = np.log(
        (Hxx_t * Z[0, 0] * np.conj(Hxx_t)).real
        * (Hyy_c * Z[1, 1] * np.conj(Hyy_c)).real
        / det_S.real
    ).real
    return Ix2y, Iy2x, Ixy


print('Helpers defined.')


Helpers defined.


## Benchmark: total pipeline time vs. N

For each N and each (spectral method × Wilson backend) combination,
we time the **complete pipeline** (CSD + Wilson/AR + GC).

JAX Wilson is warmed up (JIT-compiled) once per N before timing starts.

In [ ]:
results = defaultdict(dict)   # results[label][N] = time_s

for N in N_VALUES:
    data = make_data(N, seed=10)
    print(f'N = {N:5d}  ', end='', flush=True)

    # Parametric — no Wilson step
    results['Parametric'][N] = time_fn(lambda d=data: run_parametric(d))
    print('param', end=' ', flush=True)

    # NumPy Wilson
    for csd_name, _ in BUILDERS:
        sm = SPECTRAL_METHOD_MAP[csd_name]
        results[f'{csd_name} + NumPy'][N] = time_fn(
            lambda d=data, s=sm: run_numpy(d, s)
        )
    print('numpy', end=' ', flush=True)

    # JAX Wilson — warm up JIT for this freq_len, then time
    if JAX_AVAILABLE:
        for csd_name, _ in BUILDERS:
            sm = SPECTRAL_METHOD_MAP[csd_name]
            run_jax(data, sm)          # compilation warmup

        for csd_name, _ in BUILDERS:
            sm = SPECTRAL_METHOD_MAP[csd_name]
            results[f'{csd_name} + JAX'][N] = time_fn(
                lambda d=data, s=sm: run_jax(d, s)
            )
        print('jax', end=' ', flush=True)

    print('\u2713')

print('\nBenchmark complete.')


N =  1024  param numpy jax ✓
N =  1280  param numpy jax ✓
N =  1536  param numpy jax ✓
N =  1792  param numpy jax ✓
N =  2048  param numpy jax ✓
N =  2304  param numpy jax ✓
N =  2560  param numpy jax ✓
N =  2816  param numpy jax ✓
N =  3072  param numpy jax ✓
N =  3328  param numpy jax ✓
N =  3584  param numpy jax ✓
N =  3840  param numpy jax ✓
N =  4096  param numpy jax ✓
N =  4352  param numpy jax ✓
N =  4608  param numpy jax ✓
N =  4864  param numpy jax ✓
N =  5120  param numpy jax ✓
N =  5376  param numpy jax ✓
N =  5632  param numpy jax ✓
N =  5888  param numpy jax ✓
N =  6144  param numpy jax ✓
N =  6400  param numpy jax ✓
N =  6656  param numpy jax ✓
N =  6912  param numpy jax ✓
N =  7168  param numpy jax ✓
N =  7424  param numpy jax ✓
N =  7680  param numpy jax ✓
N =  7936  param numpy jax ✓
N =  8192  param numpy jax ✓
N =  8448  param numpy jax ✓
N =  8704  param numpy jax ✓
N =  8960  

## Results

In [ ]:
df = pd.DataFrame(results).T
df.columns.name = "N"
df.index.name   = "Method"
df_ms = (df * 1000).round(1)
df_ms.columns = [f"N={n}" for n in df_ms.columns]

print("Median wall-clock time (ms)\n")
print(df_ms.to_string())

In [ ]:
COLORS = {"FFT": "C0", "Welch": "C1", "Morlet": "C2", "Parametric": "C3"}
LS     = {"NumPy": "-",  "JAX":   "--"}
MK     = {"NumPy": "o",  "JAX":   "s"}

fig, ax = plt.subplots(figsize=(12, 5))

for label, times in results.items():
    Ns = sorted(times)
    ts = [times[n] * 1000 for n in Ns]
    if " + " in label:
        csd, backend = label.split(" + ")
        kw = dict(color=COLORS[csd], linestyle=LS[backend],
                  marker=MK[backend], lw=2, markersize=7, label=label)
    else:
        kw = dict(color=COLORS[label], linestyle=":",
                  marker="^", lw=2, markersize=7, label=label)
    ax.plot(Ns, ts, **kw)

ax.set_xscale("log", base=2)
ax.set_yscale("log")
ax.set_xticks(N_VALUES)
plt.xticks(rotation=90)
ax.set_xticklabels(N_VALUES)
ax.set_xlabel("N (time samples)")
ax.set_ylabel("Wall-clock time (ms)")
ax.set_title(
    f"Spectral GC — total pipeline time vs. data size\n"
    f"(2-ch Gaussian noise, Fs={Fs} Hz, {N_REPEATS} repeats)"
)
ax.legend(ncol=2, fontsize=10)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()